# Project 4: how much of Suriname is still forest?

**The question.** How much of Suriname's land is still forest, and how fast is that changing?

**Who it is for.** A geography teacher at a secondary school in Suriname, and the teacher's class. They get a one-page chart sheet for a lesson: an honest picture of how much of their country is forest and how fast it is being lost, a misleading chart of the same numbers with its stamp, and a short explanation of how to tell the two apart.

**The data.** Two World Bank series for Suriname, both taken from the Food and Agriculture Organization of the United Nations (FAO): forest area as a percent of land area (indicator AG.LND.FRST.ZS) and forest area in square kilometres (indicator AG.LND.FRST.K2). The World Bank last updated both series on 2026-10-08. The course file puts them side by side in one table, one row per year.

**Your team.** Four roles rotate every week: builder, data checker, AI checker and presenter. This week the data checker confirms that the two columns tell the same story, and the AI checker tests the AI's chart and its description against the numbers.

**What you hand in on Friday.** This notebook with every CHECK passing, the chart sheet for the geography class (the honest and the misleading chart side by side, the chart of how fast the forest is being lost, and about 200 words), and your AI log.

## Load the data

This cell is given. It reads the table from the course folder, or builds it straight from the World Bank if the file is not there.

In [ ]:
import json, urllib.request
import pandas as pd

URL = "https://api.worldbank.org/v2/country/SUR/indicator/{}?format=json&date=1990:2025&per_page=100"
try:
    forest = pd.read_csv("../data/week04-forest.csv")
except FileNotFoundError:
    # no course file: build the same table from the World Bank's two series
    columns = {"AG.LND.FRST.ZS": "forest_pct", "AG.LND.FRST.K2": "forest_km2"}
    parts = []
    for code, name in columns.items():
        raw = json.load(urllib.request.urlopen(URL.format(code)))[1]
        parts.append(pd.Series({int(r["date"]): r["value"] for r in raw}, name=name, dtype=float))
    forest = pd.concat(parts, axis=1).dropna(how="all").rename_axis("year").sort_index().reset_index()
forest

This cell is given too. It makes two short names for the columns, indexed by year, and brings the exaggeration factor from the skills notebook.

In [ ]:
import matplotlib.pyplot as plt

pct = forest.set_index("year")["forest_pct"]     # forest area, percent of land area
km2 = forest.set_index("year")["forest_km2"]     # forest area, square kilometres

def exaggeration(base, other, bottom):
    """How many times bigger a chart makes the change from base to other look than it is.
    bottom is where the chart's value axis starts. A chart from zero has a factor of 1."""
    real = (other - base) / base
    drawn = (other - base) / (base - bottom)
    return drawn / real

## TODO 1: describe the table

Print the number of rows, the first and last year, and the missing values in each column. Then store in `last_pct_year` and `last_km2_year` the last year that has a value in each column. The two columns do not have to end in the same year.

In [ ]:
# TODO 1
last_pct_year = None
last_km2_year = None

In [ ]:
# CHECK 1
assert last_pct_year is not None and last_km2_year is not None, "store both years"
assert last_pct_year == forest.loc[forest["forest_pct"].notna(), "year"].max()
assert last_km2_year == forest.loc[forest["forest_km2"].notna(), "year"].max()
print("CHECK 1 passed: percent to", last_pct_year, "and square kilometres to", last_km2_year)

## TODO 2: the honest chart of the level

Draw a line chart of `forest_pct` by year with the y-axis from 0 to 100 percent. Give it a title that states the claim, label both axes with their units, and name the World Bank as the source in a note on the chart. Store the axes in `honest_ax`.

In [ ]:
# TODO 2
honest_ax = None

In [ ]:
# CHECK 2
assert honest_ax is not None, "honest_ax is still empty"
lo, hi = honest_ax.get_ylim()
assert lo == 0 and hi >= 100, "the honest chart's y-axis runs from 0 to 100 percent"
assert honest_ax.get_title().strip() and honest_ax.get_ylabel().strip(), "give the chart a claim title and a y-axis label"
print("CHECK 2 passed: the axis runs from", lo, "to", hi)

## TODO 3: measure the change

Compare like with like: store in `end_year` the last year that both columns have. Store in `peak_year` the year with the most forest. Then store:

- `points_lost`: the fall in `forest_pct` from `peak_year` to `end_year`, in percentage points;
- `km2_lost`: the fall in `forest_km2` over the same years, in square kilometres;
- `share_lost`: `km2_lost` as a percent of the forest in `peak_year`;
- `points_since_1990`: the fall in `forest_pct` from 1990 to `end_year`, in percentage points.

The data checker then computes `share_lost` a second way, from the percent column, and confirms that the two columns agree. On the chart sheet, say which start year you use and why.

In [ ]:
# TODO 3
end_year = None
peak_year = None
points_lost = None
km2_lost = None
share_lost = None
points_since_1990 = None

In [ ]:
# CHECK 3
assert None not in (end_year, peak_year, points_lost, km2_lost, share_lost, points_since_1990), "fill in all six"
assert end_year == forest.dropna()["year"].max(), "end_year is the last year with a value in both columns"
assert peak_year == km2.idxmax()
assert abs(points_lost - (pct[peak_year] - pct[end_year])) < 0.001
assert abs(km2_lost - (km2[peak_year] - km2[end_year])) < 0.1
assert abs(share_lost - km2_lost / km2[peak_year] * 100) < 0.001
assert abs(points_since_1990 - (pct[1990] - pct[end_year])) < 0.001
print("CHECK 3 passed:", peak_year, "to", end_year, round(points_lost, 3), "points,", round(km2_lost, 1), "km2,", round(share_lost, 3), "percent")
print("share lost, from the percent column:", round(points_lost / pct[peak_year] * 100, 3), "percent")

## TODO 4: how fast?

Print the yearly change in square kilometres with `km2.diff()` and read it. The change stays the same for several years in a row and then steps to a new value. Write down in your AI log what that pattern tells you about how the in-between years were made.

Then store in `loss_per_year` the average number of square kilometres lost per year in each of the four periods given below, as a Series labelled `"2000-2010"`, `"2010-2015"`, `"2015-2020"` and `"2020-2025"`. Store in `speedup` how many times faster the loss was in 2020-2025 than in 2000-2010.

Draw `loss_per_year` as a bar chart with bars from zero, a claim title and labelled axes, and store its axes in `speed_ax`.

In [ ]:
# TODO 4
periods = [(2000, 2010), (2010, 2015), (2015, 2020), (2020, 2025)]
loss_per_year = None
speedup = None
speed_ax = None

In [ ]:
# CHECK 4
assert loss_per_year is not None and speedup is not None and speed_ax is not None, "fill in all three"
expected = pd.Series({f"{a}-{b}": (km2[a] - km2[b]) / (b - a) for a, b in [(2000, 2010), (2010, 2015), (2015, 2020), (2020, 2025)]})
assert [str(i) for i in loss_per_year.index] == list(expected.index), "label the periods 2000-2010, 2010-2015, 2015-2020 and 2020-2025"
assert (abs(loss_per_year.values - expected.values) < 0.1).all(), "losses are positive square kilometres per year"
assert abs(speedup - expected["2020-2025"] / expected["2000-2010"]) < 0.01
assert speed_ax.get_ylim()[0] == 0, "the bars start at zero"
print("CHECK 4 passed:", [round(float(v), 1) for v in expected.values], "km2 a year; speedup", round(speedup, 2))

## TODO 5: the misleading chart, built on purpose

Draw the honest chart and a misleading chart of the same `forest_pct` series side by side, with `plt.subplots(1, 2)`. In the misleading one, cut the y-axis so that the fall from `peak_year` to `end_year` looks at least 20 times bigger than it is, keep every point on the chart, and give it a dramatic title. Stamp it with the words *misleading on purpose* using `ax.text`.

Store the misleading chart's axes in `cut_ax` and its exaggeration factor in `cut_factor`. The misleading chart only ever appears beside the honest one, with its stamp.

In [ ]:
# TODO 5
cut_ax = None
cut_factor = None

In [ ]:
# CHECK 5
assert cut_ax is not None and cut_factor is not None, "fill in cut_ax and cut_factor"
stamp = " ".join(t.get_text().lower() for t in cut_ax.texts)
assert "misleading" in stamp or "misleidend" in stamp, "stamp the chart with the words: misleading on purpose"
bottom = cut_ax.get_ylim()[0]
assert bottom < forest["forest_pct"].min(), "keep every point on the chart"
assert abs(cut_factor - exaggeration(pct[peak_year], pct[end_year], bottom)) < 0.01
assert cut_factor >= 20, "cut the axis closer to the data: the factor must be at least 20"
print("CHECK 5 passed: the axis starts at", round(bottom, 2), "and the factor is", round(cut_factor, 1))

## TODO 6: the AI check

Ask your AI assistant, in these words: *"My table `forest` has the columns year, forest_pct (forest area in percent of land area) and forest_km2. Write matplotlib code for a line chart of forest_pct by year."*

1. Paste its code into the cell below and run it. Just before `plt.show()`, add the line `ai_ax = plt.gca()`.
2. Save the chart as an image, give it to the AI, and ask: *"In two sentences, what does this chart show about Suriname's forest?"* Write its answer in your AI log, word for word.
3. Store in `ai_factor` the exaggeration factor of the AI's chart for the fall from `peak_year` to `end_year`, using `ai_ax.get_ylim()[0]` as the bottom.
4. In the AI log, test every word of the AI's description that says how big or how fast against `share_lost` and `speedup`.

If the AI's chart starts at zero, its factor is 1, and you record that in the log too.

In [ ]:
# TODO 6
ai_ax = None
ai_factor = None
# Paste the AI's chart code below. Just before plt.show(), add: ai_ax = plt.gca()

In [ ]:
# CHECK 6
assert ai_ax is not None and ai_factor is not None, "fill in ai_ax and ai_factor"
ai_bottom = ai_ax.get_ylim()[0]
assert abs(ai_factor - exaggeration(pct[peak_year], pct[end_year], ai_bottom)) < 0.01
print("CHECK 6 passed: the AI's axis starts at", round(ai_bottom, 2), "so its chart makes the fall look", round(ai_factor, 1), "times bigger than it is")
print("The real fall from", peak_year, "to", end_year, "is", round(share_lost, 2), "percent of the forest.")

## TODO 7: the chart sheet for a geography class

Build a one-page chart sheet that a geography teacher at a secondary school in Suriname could hand to a class. Put three charts on it: the honest chart from task 2, the speed chart from task 4, and the stamped misleading chart beside its honest twin from task 5.

In a text cell below, write the sheet's text: about 200 words in plain language for secondary-school students, covering four points.

- How much of the land is forest, and the change in percentage points and in percent, with the start year named.
- How fast the loss is now compared with 2000-2010, in square kilometres per year.
- How the misleading chart tricks the eye, with its exaggeration factor, and one question a student can ask of any chart to catch the trick.
- One thing the table cannot tell the class, such as why the forest is shrinking or where.

Explain every term the class may not know, such as percentage point. The data checker reads the sheet against the notebook before Friday.